<style>
.custom-spacing {
    line-height: 0.5;
}
</style>

### <center>**KECERDASAN ARTIFISIAL UNTUK GEOFISIKA**</center>
#### <center>**TUGAS INDIVIDU**</center>

<div style="text-align:center;">
  <img src="https://www.unhas.ac.id/wp-content/uploads/2022/09/Logo-Resmi-Unhas-1.png" alt="Deskripsi Gambar" style="width:280px;">
</div>

### <center>**DISUSUN OLEH :**</center>

#### <center>**DWI REZKY MAHARANI          (H061241078)**<center>

### <center>**PROGRAM STUDI GEOFISIKA**</center>
### <center>**FAKULTAS MATEMATIKA DAN ILMU PENGETAHUAN ALAM**</center>
#### <center>**UNIVERSITAS HASANUDDIN**</center>
#### <center>**MAKASSAR**</center>
#### <center>**2026**</center>

<hr>

# Form Prompt AI untuk Geofisika (Geolistrik dan AWS)


<div align="center">

| No | Bidang | Topik Prompt |
|----|--------|--------------|
| 1 | Geolistrik | Inversi 1D sounding Schlumberger (VES) dan penentuan lapisan akuifer |
| 2 | Geolistrik | Interpretasi penampang ERT 2D untuk dugaan bidang gelincir longsor |
| 3 | AWS | QA/QC data Automatic Weather Station 10 menitan |
| 4 | AWS | Analisis pola diurnal dan variasi bulanan suhu dan curah hujan |
| 5 | Geolistrik + AWS | Integrasi curah hujan AWS dan ERT time-lapse untuk infiltrasi |

</div>

<br>
<br>

<div align="center">

| Kode | Komponen | Fungsi |
|------|----------|--------|
| **R** | Role | Sudut pandang kerja, bukan credential |
| **T** | Task | Verb yang bisa diperiksa + objek + domain + kondisi penerimaan |
| **C** | Context | Tujuan, audiens, workflow, keputusan yang didukung |
| **I** | Input | Path, format, variabel, unit, CRS/datum, provenance |
| **K** | Constraints | File yang tidak boleh diubah, method, package, jaringan, approval |
| **O** | Output | Kontrak output: path, kolom/unit, format (rencana teks, belum hasil) |
| **Q** | Quality criteria | Kriteria yang dapat dinilai eksplisit |
| **V** | Optional verification | Check + expected + tindakan bila gagal |

</div>

> Bagian **O** adalah *expected output*, bukan *actual result*. Output berupa narasi dan tabel file; gambar belum dibuat, hanya direncanakan dalam teks.

## Template Umum

```markdown
## [R] Role
Bertindaklah sebagai <sudut pandang kerja>. Fokus pada <apa yang diperiksa>.

## [T] Task
<Verb> <objek> <domain/kondisi>, lalu <verb kedua>. <Kondisi penerimaan>.

## [C] Context
<Tujuan>, <audiens>, <tahap workflow>. Hasil dipakai untuk <keputusan>, bukan untuk <hal yang tidak dilakukan>.

## [I] Input
- `<path>`: format, variabel/kolom, unit, CRS/datum, sumber

## [K] Constraints
- Jangan mengubah <file/folder>.
- Method: <yang boleh>; <yang dilarang>.
- Software: <yang tersedia>; tanpa jaringan.
- Bila <kondisi tak jelas>: berhenti dan tanyakan.

## [O] Output (rencana teks)
- `<path output>`: <isi, kolom, unit>

## [Q] Quality criteria
- <kriteria terukur>

## [V] Optional verification
- Check: <command>. Expected: <kondisi>. Jika gagal: <tindakan>.
```

---

## **Prompt 1 — Geolistrik: Inversi 1D Sounding Schlumberger (VES) dan Penentuan Akuifer**

### **[R] Role**
Bertindaklah sebagai reviewer interpretasi geolistrik untuk hidrogeologi. Fokus pada kualitas data sounding, transparansi asumsi model lapisan, dan pemisahan antara hasil inversi dan dugaan litologi.

### **[T] Task**
Hitung resistivitas semu dari data sounding Schlumberger `data/raw/ves_points.csv`, lakukan inversi 1D per titik sounding, lalu susun tabel model lapisan (ketebalan dan resistivitas) dan narasi dugaan lapisan akuifer. Jangan menetapkan titik pengeboran final.

### **[C] Context**
Survei VES dilakukan di `{{lokasi}}` pada litologi dominan `{{aluvial/batupasir/vulkanik}}` untuk mendukung rencana sumur air tanah `{{desa/irigasi}}`. Hasil dipakai mahasiswa dan tim lapangan sebagai bahan diskusi awal. Hasil tidak dipakai sebagai keputusan pengeboran tanpa korelasi data bor.

### **[I] Input**
- `data/raw/ves_points.csv`: kolom `point_id`, `AB2_m` (m), `MN2_m` (m), `I_mA` (mA), `dV_mV` (mV), `lat`, `lon` (WGS84). Sumber: resistivitymeter `{{merek/model}}`.
- Konfigurasi: Schlumberger, faktor geometri `K = π (L² − l²) / (2l)` dengan `L = AB/2` dan `l = MN/2`.
- Parameter inversi: jumlah lapisan awal `{{3/4}}`, batas resistivitas `{{min–max Ω·m}}`.
- Data pembanding (bila ada): `data/raw/borelog.csv` (`depth_m`, `litologi`).

### **[K] Constraints**
- Jangan mengubah `data/raw/`; jangan menghapus titik atau baris yang janggal, cukup beri flag.
- Jangan mengarang data atau mengisi nilai hilang dengan interpolasi.
- Semua asumsi (jumlah lapisan, model awal, batas resistivitas) ditulis eksplisit di `run_config.json`.
- Software: Python dengan NumPy dan SciPy; pyGIMLi/ResIPy hanya bila sudah terpasang; tanpa jaringan.
- Kesimpulan hidrogeologi dinyatakan sebagai indikasi, bukan kepastian; jangan menyebut kedalaman bor final.
- Bila jumlah lapisan tidak dapat dibenarkan oleh data: berhenti dan tanyakan.

### **[O] Output (rencana teks)**
- `results/ves/ves_apparent_rho.csv`: `point_id`, `AB2_m`, `K_m`, `rho_a_ohm_m`, `qc_flag`.
- `results/ves/ves_layers.csv`: `point_id`, `layer_no`, `depth_top_m`, `depth_bottom_m`, `thickness_m`, `resistivity_ohm_m`, `rms_pct`.
- `results/ves/interpretation.md`: ringkasan kualitas data, penjelasan tiap lapisan dan dugaan litologi, kandidat lapisan akuifer beserta rentang kedalaman, daftar asumsi dan keterbatasan. Grafik hanya direncanakan (kurva sounding dan model inversi), belum digambar.
- `results/ves/run_config.json`: jumlah lapisan, model awal, batas, versi package.

### **[Q] Quality criteria**
- RMS error inversi dilaporkan untuk setiap titik.
- Setiap klaim litologi disertai nilai resistivitas dan dibandingkan dengan rentang umum (mis. pasir jenuh air vs lempung).
- Ketidakpastian (prinsip ekuivalensi) disebutkan.
- Jumlah titik output = jumlah titik input (termasuk yang di-flag).
- Fakta terukur, asumsi, dan dugaan dipisahkan dalam laporan.

### **[V] Optional verification**
- Check: hitung SHA-256 `ves_points.csv` sebelum dan sesudah prosedur. Expected: identik. Jika gagal: hentikan dan laporkan.
- Check: ulangi inversi dengan 3 dan 4 lapisan pada 2 titik. Expected: lapisan akuifer utama tetap pada kedalaman serupa. Jika tidak: laporkan sebagai model tidak stabil.
- Check: bandingkan dengan `borelog.csv` bila ada. Expected: dugaan litologi konsisten. Jika tidak: catat ketidaksesuaian, jangan menyesuaikan model.

---

## **Prompt 2 — Geolistrik: Interpretasi ERT 2D untuk Dugaan Bidang Gelincir**

### **[R] Role**
Bertindaklah sebagai reviewer interpretasi ERT untuk kebencanaan. Fokus pada pembedaan fakta data, interpretasi, dan alternatif penjelasan sebelum hasil dibaca pengambil keputusan.

### **[T] Task**
Deskripsikan sebaran resistivitas pada penampang hasil inversi `data/processed/ert_inverted.csv`, identifikasi zona resistivitas rendah, menengah, dan tinggi, lalu tandai zona yang diduga bidang gelincir atau lapisan jenuh air. Jangan menetapkan tingkat bahaya longsor.

### **[C] Context**
Lereng di `{{lokasi}}` mengalami retakan setelah hujan lebat. Survei memakai konfigurasi `{{dipole-dipole/Wenner}}`, spasi elektroda `{{5}}` m, panjang lintasan `{{n}}` m. Laporan dipakai sebagai bahan awal pemerintah daerah untuk menentukan survei lanjutan. Hasil bukan analisis stabilitas lereng atau peta zona bahaya.

### **[I] Input**
- `data/processed/ert_inverted.csv`: `x_m`, `z_m` (m, z positif ke atas, datum lokal), `rho_ohm_m` (Ω·m), `rms_pct`, `doi_flag` (bila ada).
- `data/processed/topography.csv`: `x_m`, `elev_m`.
- `data/raw/field_notes.md`: kondisi lapangan, posisi retakan, curah hujan sebelum survei `{{mm}}`.

### **[K] Constraints**
- Jangan mengubah `data/processed/` dan `data/raw/`.
- Jangan menjalankan ulang inversi atau mengubah regularisasi.
- Bedakan dengan jelas fakta dari data dan interpretasi; setiap zona wajib punya rentang Ω·m dan koordinat.
- Sebutkan bila kedalaman investigasi (DOI) tidak mendukung kesimpulan.
- Ringkasan akhir memakai bahasa yang dapat dipahami non-geofisikawan.
- Bila topografi tidak konsisten dengan penampang: berhenti dan tanyakan.

### **[O] Output (rencana teks)**
- `results/ert/zones.csv`: `zone_id`, `x_min_m`, `x_max_m`, `z_min_m`, `z_max_m`, `rho_min`, `rho_max` (Ω·m), `kelas` (rendah/menengah/tinggi), `dugaan_material`, `keyakinan`.
- `results/ert/interpretation.md`: deskripsi kiri ke kanan lintasan, penjelasan zona, posisi dan kedalaman dugaan bidang gelincir, alternatif interpretasi (lempung vs air), ringkasan untuk pengambil keputusan, saran survei lanjutan. Penampang hanya direncanakan dalam teks, belum digambar.
- `results/ert/run_config.json`: batas kelas resistivitas dan alasan pemilihannya.

### **[Q] Quality criteria**
- Batas kelas resistivitas dinyatakan dan diberi alasan.
- Interpretasi konsisten dengan topografi dan catatan lapangan.
- Minimal satu alternatif interpretasi dibahas untuk setiap zona penting.
- Zona di luar DOI ditandai tidak dapat diinterpretasi.
- Tidak ada pernyataan tingkat bahaya definitif.

### **[V] Optional verification**
- Check: hitung ulang rentang Ω·m dua zona langsung dari `ert_inverted.csv`. Expected: sama dengan `zones.csv`. Jika gagal: koreksi tabel dan laporkan.
- Check: bandingkan posisi dugaan bidang gelincir dengan lokasi retakan di `field_notes.md`. Expected: berdekatan atau dijelaskan bila tidak. Jika tidak: tulis sebagai ketidaksesuaian.

---

## **Prompt 3 — AWS: QA/QC Data Automatic Weather Station 10 Menitan**

### **[R] Role**
Bertindaklah sebagai reviewer QC data meteorologi. Fokus pada preservasi data mentah, timestamp, satuan, dan transparansi ambang QC.

### **[T] Task**
Periksa data AWS `data/raw/aws_10min.csv` untuk nilai di luar rentang fisik, lonjakan tidak wajar, data beku (flatline), timestamp duplikat, dan data hilang, lalu tulis laporan QC dengan kolom flag terpisah. Jangan mengisi data hilang.

### **[C] Context**
Data AWS di `{{lokasi}}` periode `{{tanggal awal–akhir}}` akan dipakai untuk analisis iklim lokal dan validasi model cuaca. Laporan QC dipakai untuk memutuskan data mana yang layak dianalisis. Hasil tidak dipakai untuk analisis tren atau interpretasi iklim.

### **[I] Input**
- `data/raw/aws_10min.csv`: `timestamp` (zona waktu `{{WIB/UTC}}`), `temp_c` (°C), `rh_pct` (%), `pres_hpa` (hPa), `ws_ms` (m/s), `wd_deg` (derajat dari utara), `rad_wm2` (W/m²), `rain_mm` (mm per 10 menit).
- `data/raw/station_meta.json`: `lat`, `lon`, `elev_m`, tinggi sensor `{{m}}`.
- Ambang QC: `{{suhu min/max, RH 0–100, tekanan min/max, lonjakan maksimum per 10 menit, durasi flatline}}`.

### **[K] Constraints**
- Jangan mengubah `data/raw/`; jangan menghapus baris; flag disimpan pada kolom terpisah.
- Jangan mengisi data hilang (tanpa interpolasi atau imputasi).
- Setiap ambang ditulis eksplisit beserta alasannya; jangan menetapkan ambang tanpa menyebutnya.
- Jika zona waktu tidak jelas, jangan menebak: berhenti dan tanyakan.
- Software: Python dengan pandas dan NumPy; tanpa jaringan.

### **[O] Output (rencana teks)**
- `results/aws_qc/aws_10min_flagged.csv`: semua kolom asli ditambah `flag_range`, `flag_spike`, `flag_flatline`, `flag_missing`, `flag_duplicate`.
- `results/aws_qc/qc_report.md`: jumlah data, persentase data hilang per parameter, jumlah dan contoh waktu tiap jenis anomali, ambang beserta alasan, rekomendasi data layak/perlu dicek/dibuang.
- `results/aws_qc/summary.json`: `n_rows`, `n_flagged` per jenis, `pct_missing` per parameter.
- `results/aws_qc/run_config.json`: ambang dan versi package.

### **[Q] Quality criteria**
- Jumlah baris output = jumlah baris input.
- Persentase dihitung dari total baris input yang benar.
- Setiap flag memuat parameter, nilai, ambang, dan alasan.
- Metode dapat direproduksi dari `run_config.json`.
- Data mentah tidak berubah.

### **[V] Optional verification**
- Check: hitung SHA-256 `aws_10min.csv` sebelum dan sesudah. Expected: identik. Jika gagal: hentikan dan laporkan.
- Check: periksa manual 20 baris berflag acak. Expected: flag sesuai aturan. Jika ada yang salah: laporkan aturan yang bermasalah.
- Check: bandingkan sebagian hasil dengan stasiun terdekat `{{nama stasiun}}` bila ada. Expected: pola serupa. Jika tidak: catat sebagai temuan, bukan koreksi.

---

## **Prompt 4 — AWS: Pola Diurnal dan Variasi Bulanan Suhu dan Curah Hujan**

### **[R] Role**
Bertindaklah sebagai reviewer analisis klimatologi deskriptif. Fokus pada ukuran sampel, statistik yang sesuai, dan batas kesimpulan yang boleh ditarik dari data.

### **[T] Task**
Hitung siklus diurnal suhu dan curah hujan serta variasi bulanan dari data AWS yang sudah lolos QC `data/processed/aws_clean.csv`, lalu jelaskan hasilnya secara fisis. Jangan menyimpulkan tren jangka panjang.

### **[C] Context**
Lokasi di wilayah tropis `{{pesisir/dataran tinggi}}` dengan pengaruh monsun. Hasil dipakai untuk bab pembahasan tugas akhir. Hasil bukan analisis perubahan iklim dan bukan prakiraan cuaca.

### **[I] Input**
- `data/processed/aws_clean.csv`: `timestamp` (`{{WIB/UTC}}`), `temp_c` (°C), `rain_mm` (mm per 10 menit), `qc_ok` (boolean). Periode `{{minimal 1 tahun}}`.
- Definisi musim: `{{mis. hujan Nov–Mar, kemarau Mei–Sep}}`, sumber `{{BMKG/literatur}}`.
- Ambang hujan ekstrem: `{{mm/hari}}`.

### **[K] Constraints**
- Jangan mengubah `data/processed/`; baris dengan `qc_ok = False` dikeluarkan dan jumlahnya dilaporkan.
- Gunakan rata-rata, median, dan persentil; jangan hanya rata-rata.
- Jangan menarik kesimpulan tren jangka panjang dari data kurang dari 10 tahun; tulis keterbatasannya.
- Bedakan korelasi dan sebab-akibat.
- Software: Python dengan pandas dan NumPy; tanpa jaringan.

### **[O] Output (rencana teks)**
- `results/aws_diurnal/diurnal_stats.csv`: `jam`, `temp_mean`, `temp_median`, `temp_p25`, `temp_p75`, `rain_mean_mm`, `n` (per jam lokal).
- `results/aws_diurnal/monthly_stats.csv`: `bulan`, `temp_mean`, `rain_total_mm`, `n_hari_valid`, `n_hari_hujan`.
- `results/aws_diurnal/analysis.md`: jam puncak suhu dan hujan, perbandingan musim hujan dan kemarau, hari hujan ekstrem, interpretasi fisis (pemanasan siang, konveksi, monsun), keterbatasan. Grafik hanya direncanakan (mis. boxplot per jam), belum digambar.
- `results/aws_diurnal/run_config.json`: zona waktu, definisi musim, ambang ekstrem.

### **[Q] Quality criteria**
- Setiap angka disertai satuan dan jumlah sampel `n`.
- Jumlah data yang dikeluarkan karena QC dilaporkan.
- Interpretasi fisis sesuai teori dasar meteorologi dan tidak melampaui data.
- Zona waktu tercatat dan konsisten.

### **[V] Optional verification**
- Check: hitung ulang median suhu pada 3 jam berbeda secara independen. Expected: sama dengan `diurnal_stats.csv`. Jika gagal: laporkan perbedaan.
- Check: bandingkan pola bulanan dengan normal klimatologi `{{BMKG wilayah terdekat}}`. Expected: pola musim serupa. Jika tidak: catat dan diskusikan kemungkinan penyebab.

---

## **Prompt 5 — Geolistrik + AWS: Curah Hujan dan ERT Time-Lapse untuk Infiltrasi**

### **[R] Role**
Bertindaklah sebagai reviewer analisis terpadu geofisika-meteorologi. Fokus pada penyelarasan waktu, definisi akumulasi hujan, dan kehati-hatian dalam klaim kausal.

### **[T] Task**
Hitung akumulasi curah hujan AWS sebelum setiap tanggal survei, hitung perubahan resistivitas relatif terhadap survei baseline pada lintasan ERT yang sama, lalu jelaskan apakah pola perubahan konsisten dengan infiltrasi hujan. Jangan menyimpulkan hubungan sebab-akibat.

### **[C] Context**
ERT diukur pada lintasan yang sama di `{{lokasi}}` sebanyak `{{3–4}}` kali (mis. akhir kemarau, awal hujan, puncak hujan). AWS terdekat berjarak sekitar `{{x}}` km. Hasil dipakai untuk studi resapan dan kerentanan lereng pada tahap eksplorasi. Hasil bukan model infiltrasi dan bukan penilaian stabilitas lereng.

### **[I] Input**
- `data/processed/aws_daily.csv`: `date`, `rain_mm` (mm/hari), `temp_c` (°C), zona waktu `{{WIB/UTC}}`.
- `data/processed/ert_t1.csv`, `ert_t2.csv`, `ert_t3.csv`: `x_m`, `z_m` (m), `rho_ohm_m` (Ω·m), grid inversi identik.
- `data/raw/survey_log.csv`: `survey_id`, `date`, `kondisi_tanah`, `rms_pct`.
- Jendela akumulasi: `{{3, 7, 14}}` hari. Survei baseline: `{{survey_id}}`.

### **[K] Constraints**
- Jangan mengubah data input; jangan menjalankan ulang inversi.
- Jika grid antar survei tidak identik, jangan resampling diam-diam: berhenti dan tanyakan.
- Perubahan resistivitas dinyatakan sebagai persen relatif terhadap baseline: `(rho_t − rho_0) / rho_0 × 100`.
- Jangan mengklaim kausalitas; bahas faktor lain (suhu tanah, litologi, kualitas inversi, jarak AWS).
- Software: Python dengan pandas dan NumPy; tanpa jaringan.

### **[O] Output (rencana teks)**
- `results/timelapse/rain_accum.csv`: `survey_id`, `date`, `accum_3d_mm`, `accum_7d_mm`, `accum_14d_mm`, `n_missing_days`.
- `results/timelapse/rho_change.csv`: `survey_id`, `x_m`, `z_m`, `rho_ohm_m`, `delta_pct`.
- `results/timelapse/analysis.md`: perubahan resistivitas per kedalaman dan zona, hubungan dengan akumulasi hujan, zona yang tidak merespons beserta dugaan penyebab, keterbatasan, rekomendasi pemantauan lanjutan. Peta perubahan dan grafik hujan vs resistivitas hanya direncanakan, belum digambar.
- `results/timelapse/run_config.json`: jendela akumulasi, baseline, zona waktu.

### **[Q] Quality criteria**
- Setiap kesimpulan menyebut data pendukung (akumulasi hujan dan persentase perubahan).
- Hari hujan hilang dilaporkan dan tidak diisi.
- Alternatif penjelasan dibahas untuk setiap zona yang berubah.
- Pilihan jendela akumulasi dinyatakan dan diberi alasan.
- Struktur laporan konsisten dari data ke interpretasi ke kesimpulan.

### **[V] Optional verification**
- Check: hitung ulang `accum_7d_mm` satu survei secara manual. Expected: sama dengan `rain_accum.csv`. Jika gagal: laporkan kesalahan penjumlahan.
- Check: ulangi analisis dengan jendela 3 dan 14 hari. Expected: arah hubungan tidak berbalik. Jika berbalik: laporkan sebagai hasil sensitif terhadap jendela.
- Check: bandingkan dengan data kelembapan tanah atau muka air sumur `{{bila tersedia}}`. Expected: arah perubahan sejalan. Jika tidak: catat sebagai ketidaksesuaian.